# CLIP + GPT-2 Captioning — Experiment Hub

This notebook is the central place to run, compare, and analyse all experiments for the **M3 ablation suite** and any custom hyperparameter searches.

## What's inside
| Section | Purpose |
|---------|----------|
| **Setup & Config** | One place to change dataset size, LR, epochs, etc. |
| **Run definitions** | The 7 predefined M3 ablation runs |
| **Helper functions** | `run_experiment`, `show_samples`, `plot_training_curve`, `compare_runs` |
| **M3 ablation cells** | One cell per run — execute and inspect inline |
| **Results comparison** | Side-by-side table + bar chart across all 7 runs |
| **Free-form section** | Template for custom hyperparameter experiments |
| **Fine-tuning advice** | What to look for, what to try next |

## Quick-start
1. Set `MAX_SAMPLES = 200` and `DEFAULT_EPOCHS = 1` in **Cell 2** for a ~3-minute smoke test.
2. Once the pipeline works end-to-end, reset `MAX_SAMPLES = -1` (full 31 k images) and run all 7 M3 cells sequentially.
3. Fill in **run_007** after reviewing the results of runs 1–6.



In [ ]:
# ── Colab: mount Drive + authenticate HuggingFace ────────────────────────────
from google.colab import userdata
import os, logging, sys
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace: authenticated")

# Mirror all logger.info() calls from train.py to notebook stdout
import logging, sys
from transformers import logging as hf_logging

# ── Silence all noisy third-party libraries ───────────────────────
logging.basicConfig(level=logging.WARNING, force=True)
hf_logging.set_verbosity_error()  # hides model load reports + HTTP info

for noisy_lib in [
    "huggingface_hub", "datasets", "urllib3",
    "filelock", "fsspec", "PIL", "httpx",
    "src.preprocessor", "src.data_loader",
]:
    logging.getLogger(noisy_lib).setLevel(logging.ERROR)

# ── Only show our own train.py logs ──────────────────────────────
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter("%(message)s"))

for our_logger in ["train", "src.decoder", "src.model"]:
    lg = logging.getLogger(our_logger)
    lg.setLevel(logging.INFO)
    lg.addHandler(_handler)
    lg.propagate = False   # prevent double-printing

print("Logging configured — only training progress will be shown")



In [ ]:
# Cell 1: Setup & Imports
import sys
import json
import argparse
import gc
import random
import warnings
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore", category=UserWarning)

# Detect environment
IN_COLAB = "google.colab" in sys.modules
print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    # Clone from GitHub
    import subprocess
    GITHUB_URL = "https://github.com/Scofe-C/Generative_Project_1"

    if not Path("/content/Generative_Project_1").exists():
        subprocess.run(["git", "clone", GITHUB_URL, "/content/Generative_Project_1"], check=True)
    else:
        # Pull latest changes if already cloned
        subprocess.run(["git", "-C", "/content/Generative_Project_1", "pull"], check=True)

    PROJECT_ROOT = Path("/content/Generative_Project_1")

    # Install only packages missing from Colab (avoids re-installing pre-installed ones)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", str(PROJECT_ROOT / "requirements" / "Colab.txt")],
        check=True,
    )
    print("Cloned + dependencies installed.")

else:
    # Local: walk up to find train.py
    _candidate = Path.cwd()
    for _ in range(4):
        if (_candidate / "train.py").exists():
            break
        _candidate = _candidate.parent
    else:
        raise FileNotFoundError("Could not find project root.")
    PROJECT_ROOT = _candidate

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root : {PROJECT_ROOT}")



from train import (
    build_dataset,
    load_models,
    apply_finetuning,
    build_optimiser,
    train_one_epoch,
    evaluate,
    save_checkpoint,
    write_run_outputs,
    make_collate_fn,
    CaptionDataset,
    composite_score,
)
from src.utils import get_device, load_config, set_seed, setup_logging

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#  Cell 2: Global Config
# Edit these values to control all experiments.
# For a quick smoke test: set MAX_SAMPLES=200 and DEFAULT_EPOCHS=1.

# Dataset
MAX_SAMPLES = -1       # -1 = full 31 783 images; 200 = smoke test
STRATIFY = False     # False = full dataset (recommended)
                    # True  = 10-class stratified (fast debugging)
SEED = 42

# Training defaults (overridden per run in M3_RUNS / CUSTOM_RUN)
DEFAULT_EPOCHS = 15
DEFAULT_BATCH_SIZE = 32    # A100 80GB; effective batch = BS * GRAD_ACCUM
DEFAULT_LR = 1e-4          # prefix_proj gets 10x via separate param groups
MAX_NEW_TOKENS = 40        # Reduced from 50 to prevent rambling
BEAM_WIDTH = 5

# Training optimizations
GRAD_ACCUM_STEPS = 2   # Effective batch = DEFAULT_BATCH_SIZE * GRAD_ACCUM_STEPS (e.g. 32*2=64)
PATIENCE         = 5   # Early stopping: stop if composite metric does not improve for this many epochs

# Evaluation controls
EVAL_EVERY   = 1    # evaluate every epoch (fast greedy for intermediate)
EVAL_SAMPLES = 500    # intermediate eval subset; full val for final eval
EVAL_BATCH_SIZE    = 128   # large eval batch for A100 (no gradients needed)

# Output root — always local (download weights manually if needed)
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f"Outputs dir  : {OUTPUTS}")

#Load master config and apply dataset overrides
config = load_config()
config["dataset"]["stratify"]  = STRATIFY
config["dataset"]["max_samples"] = MAX_SAMPLES

device = get_device()
set_seed(SEED)

print(f"Device       : {device}")
print(f"Dataset mode : {'full (non-stratified)' if not STRATIFY else '10-class stratified'}")
print(f"Max samples  : {'all' if MAX_SAMPLES < 0 else MAX_SAMPLES}")
print(f"Batch size   : {DEFAULT_BATCH_SIZE} (eff={DEFAULT_BATCH_SIZE * GRAD_ACCUM_STEPS}) | LR: {DEFAULT_LR} | Epochs: {DEFAULT_EPOCHS}")

In [ ]:
# Cell 3: Staged Hyperparameter Sweep — CLIP-L/14 + GPT-2 Medium
# Strategy: staged (not full cartesian) sweep.
# Each stage fixes the best settings from the prior stage and varies one dimension.
#
# Base config: CLIP-L/14, GPT-2 Medium, LoRA, full Flickr30k, batch=64,
#              grad_accum=2, label_smoothing=0.1, patience=5

# ── STAGE 1: Learning Rate ──────────────────────────────────────────────────
STAGE1_RUNS = [
    dict(run_id="sweep_lr_1e4", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="beam", lora_rank=16, epochs=15, lr=1e-4,
         label_smoothing=0.1, patience=5, description="Stage1 | LR=1e-4"),
    dict(run_id="sweep_lr_5e5", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="beam", lora_rank=16, epochs=15, lr=5e-5,
         label_smoothing=0.1, patience=5, description="Stage1 | LR=5e-5"),
    dict(run_id="sweep_lr_1e5", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="beam", lora_rank=16, epochs=15, lr=1e-5,
         label_smoothing=0.1, patience=5, description="Stage1 | LR=1e-5"),
]

# ── STAGE 2: LoRA Rank ──────────────────────────────────────────────────────
# >>> UPDATE BEST_LR after running Stage 1 compare_runs <<<
BEST_LR = 1e-4

STAGE2_RUNS = [
    dict(run_id="sweep_lora_r4",  encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="beam", lora_rank=4,  epochs=15, lr=BEST_LR,
         label_smoothing=0.1, patience=5, description="Stage2 | LoRA r=4"),
    dict(run_id="sweep_lora_r8",  encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="beam", lora_rank=8,  epochs=15, lr=BEST_LR,
         label_smoothing=0.1, patience=5, description="Stage2 | LoRA r=8"),
    dict(run_id="sweep_lora_r16", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="beam", lora_rank=16, epochs=15, lr=BEST_LR,
         label_smoothing=0.1, patience=5, description="Stage2 | LoRA r=16"),
]

# ── STAGE 3: Decoding Strategy ──────────────────────────────────────────────
# >>> UPDATE BEST_RANK after running Stage 2 compare_runs <<<
BEST_RANK = 16

STAGE3_RUNS = [
    dict(run_id="sweep_dec_greedy",    encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="greedy",
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage3 | Greedy"),
    dict(run_id="sweep_dec_beam",      encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="beam", beam_width=5,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage3 | Beam k=5"),
    dict(run_id="sweep_dec_nuc_07_09", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="nucleus", temperature=0.7, top_p=0.9,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage3 | Nucleus temp=0.7 top_p=0.9"),
    dict(run_id="sweep_dec_nuc_10_09", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="nucleus", temperature=1.0, top_p=0.9,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage3 | Nucleus temp=1.0 top_p=0.9"),
    dict(run_id="sweep_dec_nuc_05_095",encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="nucleus", temperature=0.5, top_p=0.95,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage3 | Nucleus temp=0.5 top_p=0.95"),
]

# ── STAGE 4: Nucleus Fine-Tuning (run only if nucleus wins Stage 3) ─────────
STAGE4_RUNS = [
    dict(run_id="sweep_nuc_05_08",  encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="nucleus", temperature=0.5, top_p=0.8,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage4 | Nucleus temp=0.5 top_p=0.8"),
    dict(run_id="sweep_nuc_05_09",  encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="nucleus", temperature=0.5, top_p=0.9,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage4 | Nucleus temp=0.5 top_p=0.9"),
    dict(run_id="sweep_nuc_07_08",  encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="nucleus", temperature=0.7, top_p=0.8,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage4 | Nucleus temp=0.7 top_p=0.8"),
    dict(run_id="sweep_nuc_10_095", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder="nucleus", temperature=1.0, top_p=0.95,
         lora_rank=BEST_RANK, epochs=15, lr=BEST_LR, label_smoothing=0.1, patience=5,
         description="Stage4 | Nucleus temp=1.0 top_p=0.95"),
]

# ── STAGE 5: Final Best Config — Extended Training ──────────────────────────
# >>> UPDATE BEST_DECODER/BEST_TEMP/BEST_TOP_P after Stage 3/4 <<<
BEST_DECODER = "beam"
BEST_TEMP    = 1.0
BEST_TOP_P   = 0.9

STAGE5_RUNS = [
    dict(run_id="sweep_final", encoder="openai/clip-vit-large-patch14", gpt2="gpt2-medium",
         finetune="lora", decoder=BEST_DECODER, temperature=BEST_TEMP, top_p=BEST_TOP_P,
         lora_rank=BEST_RANK, epochs=20, patience=7, lr=BEST_LR, label_smoothing=0.1,
         description="Stage5 | Final best config — 20 epochs, patience=7"),
]

ALL_RUNS = STAGE1_RUNS + STAGE2_RUNS + STAGE3_RUNS + STAGE4_RUNS + STAGE5_RUNS

print('Staged sweep ready:')
print(f'  Stage 1 (LR):      {len(STAGE1_RUNS)} runs')
print(f'  Stage 2 (rank):    {len(STAGE2_RUNS)} runs')
print(f'  Stage 3 (decoder): {len(STAGE3_RUNS)} runs')
print(f'  Stage 4 (nucleus): {len(STAGE4_RUNS)} runs  [conditional on Stage3 winner]')
print(f'  Stage 5 (final):   {len(STAGE5_RUNS)} run')
print(f'  Total worst-case:  {len(ALL_RUNS)} runs')


In [ ]:
# Cell 4: make_args() — run dict → argparse.Namespace
# Converts a run config dict into the Namespace expected by all train.py functions.

def make_args(run_cfg: dict) -> argparse.Namespace:
    """
    Build an argparse.Namespace from a run config dict.
    Any key in run_cfg overrides the corresponding default.
    """
    defaults = dict(
        gpt2 = "gpt2",
        injection = "prefix",
        num_prefix = 20,
        proj_depth = 3,
        proj_dropout = 0.1,
        epochs = DEFAULT_EPOCHS,
        batch_size = DEFAULT_BATCH_SIZE,
        lr = DEFAULT_LR,
        max_samples  = MAX_SAMPLES,
        seed = SEED,
        beam_width = BEAM_WIDTH,
        temperature = 1.0,
        top_p = 0.9,
        max_new_tokens = MAX_NEW_TOKENS,
        resume = None,
        label_smoothing = 0.1,
        repetition_penalty = 1.5,
        patience = PATIENCE,
        grad_accum_steps = GRAD_ACCUM_STEPS,
        eval_batch_size = EVAL_BATCH_SIZE,
        # Fields required by write_run_outputs / evaluate
        finetune  = "frozen",
        decoder = "greedy",
        lora_rank  = 16,
        encoder  = "openai/clip-vit-base-patch32",
        run_id  = "unnamed",
    )
    merged = {**defaults, **{k: v for k, v in run_cfg.items() if k != "description"}}
    return argparse.Namespace(**merged)

In [ ]:
# Cell 5: run_experiment() — full training loop

def run_experiment(
    run_cfg : dict,
    config  : dict,
    device  : torch.device,
) -> dict:
    """
    Execute a single training experiment defined by run_cfg.

    Calls train.py functions directly (no subprocess) so tensors and logs
    are fully accessible in the notebook.

    Returns a dict with:
        run_id       : str
        scores       : final metric dict {bleu_4, cider, rouge_l, ...}
        hypotheses   : {image_id: [generated_caption]}
        references   : {image_id: [ref1, ref2, ...]}
        train_log    : list of per-epoch dicts
        description  : human-readable run label
    """
    args = make_args(run_cfg)
    desc = run_cfg.get("description", args.run_id)
    print(f"\n{'='*60}")
    print(f"Starting: {args.run_id} — {desc}")
    print(f"  encoder={args.encoder} | finetune={args.finetune}"
          f" | decoder={args.decoder} | lora_rank={args.lora_rank}"
          f" | epochs={args.epochs} | lr={args.lr}")
    print(f"  eval: greedy/{EVAL_SAMPLES} intermediate, {args.decoder}/full final")
    print(f"{'='*60}")

    set_seed(args.seed)

    # Clear duplicate log handlers (Colab adds its own; prevent 2x/3x output)
    import logging as _logging
    _logging.getLogger().handlers.clear()
    setup_logging(config)

    run_dir    = OUTPUTS / "runs"    / args.run_id
    weight_dir = OUTPUTS / "weights" / args.run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    weight_dir.mkdir(parents=True, exist_ok=True)

    # 1. Dataset
    print("[1/4] Building dataset and computing CLIP embeddings...")
    cache_dir = OUTPUTS / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    config["output"]["cache_dir"] = str(cache_dir)
    train_samples, val_samples = build_dataset(args, config, device)
    print(f"      {len(train_samples)} train / {len(val_samples)} val samples")

    # 2. Models
    print("[2/4] Loading models...")
    gpt2_model, prefix_proj, tokenizer = load_models(args, config, device)
    gpt2_model, _ = apply_finetuning(gpt2_model, args)

    # Optionally load CLIP for CLIP-sim metric
    clip_model_eval, clip_proc_eval = None, None
    try:
        from transformers import CLIPModel, CLIPProcessor
        clip_model_eval = CLIPModel.from_pretrained(args.encoder).to(device).eval()
        clip_proc_eval  = CLIPProcessor.from_pretrained(args.encoder)
    except Exception:
        pass

    # 3. Training loop
    print("[3/4] Training...")
    collate_fn   = make_collate_fn(
        tokenizer,
        max_length=config["preprocessing"]["max_token_length"],
    )
    train_loader = DataLoader(
        CaptionDataset(train_samples),
        batch_size  = args.batch_size,
        shuffle     = True,
        collate_fn  = collate_fn,
        num_workers = 0 if not IN_COLAB else min(4, __import__("multiprocessing").cpu_count() // 2),
        pin_memory  = device.type == "cuda",
    )

    steps_per_epoch          = max(1, len(train_loader) // args.grad_accum_steps)
    n_train_steps            = steps_per_epoch * args.epochs
    optimizer, scheduler     = build_optimiser(gpt2_model, prefix_proj, args, n_train_steps)

    best_composite            = -1.0
    best_val_loss             = float("inf")
    epochs_no_improve         = 0
    train_log: list[dict]     = []
    final_scores: dict        = {}
    final_hypotheses: dict    = {}
    final_references: dict    = {}

    for epoch in range(1, args.epochs + 1):
        print(f"  Epoch {epoch}/{args.epochs}", end="", flush=True)
        train_loss = train_one_epoch(
            epoch, train_loader, gpt2_model, prefix_proj,
            optimizer, scheduler, tokenizer, device,
            label_smoothing=args.label_smoothing,
            grad_accum_steps=args.grad_accum_steps,
        )

        is_last_epoch = (epoch == args.epochs)
        should_eval = is_last_epoch or (epoch % EVAL_EVERY == 0)
        if should_eval:
            fast = not is_last_epoch
            eval_set = val_samples[:EVAL_SAMPLES] if (fast and EVAL_SAMPLES > 0) else val_samples
            scores, hypotheses, references = evaluate(
                eval_set, gpt2_model, prefix_proj, tokenizer, device, args,
                clip_model_eval if not fast else None,
                clip_proc_eval if not fast else None,
                eval_batch_size=EVAL_BATCH_SIZE,
                override_decoder="greedy" if fast else None,
            )
        else:
            scores     = {"bleu_4": 0.0, "cider": 0.0, "rouge_l": 0.0, "meteor": 0.0}
            hypotheses = {}
            references = {}

        did_eval = should_eval
        epoch_composite = composite_score(scores) if did_eval else 0.0
        val_loss = 1.0 - epoch_composite
        if did_eval:
            mode_str = "greedy/fast" if not is_last_epoch else f"{args.decoder}/full"
            print(f" | loss={train_loss:.4f} | BLEU-4={scores.get('bleu_4'):.4f}"
                  f" | CIDEr={scores.get('cider'):.4f} | comp={epoch_composite:.4f} [{mode_str}]")
        else:
            print(f" | loss={train_loss:.4f} | (eval skipped)")

        train_log.append({
            "epoch"      : epoch,
            "train_loss" : round(train_loss, 4),
            "bleu_4"     : scores.get("bleu_4",  -1) if did_eval else None,
            "cider"      : scores.get("cider",   -1) if did_eval else None,
            "rouge_l"    : scores.get("rouge_l", -1) if did_eval else None,
            "meteor"     : scores.get("meteor",  -1) if did_eval else None,
            "composite"  : round(epoch_composite, 4) if did_eval else None,
            "lr"         : round(scheduler.get_last_lr()[0], 8),
        })

        # Only update best / early-stop counter when eval actually ran
        if did_eval:
            is_best = epoch_composite > best_composite
            if is_best:
                best_composite    = epoch_composite
                best_val_loss     = val_loss
                epochs_no_improve = 0
                final_scores      = scores
                final_hypotheses  = hypotheses
                final_references  = references
            else:
                epochs_no_improve += 1
        else:
            is_best = False

        save_checkpoint(
            weight_dir, epoch, prefix_proj, gpt2_model, val_loss, args, is_best,
            optimizer=optimizer, scheduler=scheduler,
        )

        # Early stopping — only check after an eval epoch
        patience = getattr(args, "patience", 5)
        if did_eval and patience > 0 and epochs_no_improve >= patience:
            print(f" | Early stopping (no improvement for {epochs_no_improve} eval epochs)")
            break

    # Re-evaluate best checkpoint with full eval (actual decoder + all val)
    best_ckpt_path = weight_dir / "checkpoint_best.pt"
    if best_ckpt_path.exists():
        print("  Full eval on best checkpoint...")
        ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
        prefix_proj.load_state_dict(ckpt["prefix_proj"])
        if ckpt.get("lora_adapter") is not None:
            gpt2_model.load_state_dict(ckpt["lora_adapter"], strict=False)
        final_scores, final_hypotheses, final_references = evaluate(
            val_samples, gpt2_model, prefix_proj, tokenizer, device, args,
            clip_model_eval, clip_proc_eval,
            eval_batch_size=EVAL_BATCH_SIZE,
        )
    elif not final_scores:
        final_scores     = scores
        final_hypotheses = hypotheses
        final_references = references

    # 4. Save outputs
    print("[4/4] Writing run outputs...")
    write_run_outputs(
        run_dir, final_scores, args,
        final_hypotheses, final_references, train_log,
    )

    print(f"\nDone: {args.run_id}")
    print(f"  BLEU-4={final_scores.get('bleu_4', -1):.4f}"
          f" | CIDEr={final_scores.get('cider', -1):.4f}"
          f" | ROUGE-L={final_scores.get('rouge_l', -1):.4f}"
          f" | Composite={composite_score(final_scores):.4f}")

    # Free GPU memory before next run
    del gpt2_model, prefix_proj, train_loader, optimizer, scheduler
    if clip_model_eval is not None:
        del clip_model_eval
    if device.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()


    return {
        "run_id"      : args.run_id,
        "description" : desc,
        "scores"      : final_scores,
        "hypotheses"  : final_hypotheses,
        "references"  : final_references,
        "train_log"   : train_log,
    }

In [ ]:
# Cell 6: show_samples() — qualitative comparison with actual images

def show_samples(result: dict, n: int = 5) -> None:
    """
    Display generated vs reference captions alongside the actual images
    for the first n val samples. Images are streamed from the dataset.
    Per-image BLEU-4, ROUGE-L, and METEOR are computed live.
    """
    import matplotlib.gridspec as gridspec
    from datasets import load_dataset
    from src.metrics import compute_single_sample_metrics
    from rouge_score import rouge_scorer as rouge_lib

    run_id      = result["run_id"]
    hypotheses  = result["hypotheses"]
    references  = result["references"]

    # Collect target image IDs
    target_ids = set(list(hypotheses.keys())[:n])

    # Stream dataset to find matching images
    ds_cfg = config["dataset"]
    ds = load_dataset(
        ds_cfg["name"],
        split=ds_cfg["split"],
        revision=ds_cfg.get("revision"),
        streaming=True,
    )
    images = {}
    for sample in ds:
        img_id = str(sample.get("image_id", sample.get("img_id", sample.get("id", ""))))
        if img_id in target_ids:
            images[img_id] = sample["image"]
            if len(images) >= len(target_ids):
                break

    print(f"\n{'='*70}")
    print(f"Sample captions — {run_id}: {result.get('description', '')}")
    print(f"{'='*70}")

    # Reuse one RougeScorer across all samples for efficiency
    scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)

    for img_id in list(hypotheses.keys())[:n]:
        gen_list  = hypotheses[img_id]
        generated = gen_list[0]
        ref_list  = references.get(img_id, [])

        # Compute per-image metrics live
        if ref_list:
            per_img = compute_single_sample_metrics(generated, ref_list, rouge_scorer=scorer)
            bleu4   = per_img["bleu_4"]
            rouge   = per_img["rouge_l"]
            meteor  = per_img["meteor"]
            score_str = (
                f"BLEU-4={bleu4:.4f}  ROUGE-L={rouge:.4f}  "
                f"METEOR={'N/A' if meteor < 0 else f'{meteor:.4f}'}"
            )
        else:
            score_str = "BLEU-4=N/A  ROUGE-L=N/A  METEOR=N/A"

        # Build figure: image on left, captions on right
        fig = plt.figure(figsize=(14, 4))
        gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2], wspace=0.05)

        # Image panel
        ax_img = fig.add_subplot(gs[0])
        if img_id in images:
            ax_img.imshow(images[img_id])
        else:
            ax_img.text(0.5, 0.5, "Image not found", ha="center", va="center",
                        fontsize=12, transform=ax_img.transAxes)
        ax_img.set_title(f"Image {img_id}", fontsize=10, fontweight="bold")
        ax_img.axis("off")

        # Caption panel
        ax_txt = fig.add_subplot(gs[1])
        ax_txt.axis("off")

        caption_text = f"Generated: {generated}\n\n"
        for i, ref in enumerate(ref_list[:3], 1):
            caption_text += f"Ref {i}: {ref}\n"
        caption_text += f"\n{score_str}"

        ax_txt.text(0.02, 0.95, caption_text, transform=ax_txt.transAxes,
                    fontsize=9, verticalalignment="top", fontfamily="monospace",
                    wrap=True,
                    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f0f0",
                              edgecolor="#cccccc", alpha=0.9))

        fig.patch.set_facecolor("white")
        fig.tight_layout()
        plt.show()
        plt.close(fig)

    print(f"\n{'='*70}")
    scores = result["scores"]
    meteor_corpus = scores.get("meteor", -1)
    print(
        f"Corpus metrics — BLEU-4={scores.get('bleu_4', -1):.4f}"
        f"  CIDEr={scores.get('cider', -1):.4f}"
        f"  ROUGE-L={scores.get('rouge_l', -1):.4f}"
        f"  METEOR={'N/A' if meteor_corpus < 0 else f'{meteor_corpus:.4f}'}"
    )

In [ ]:
# Cell 7: plot_training_curve()
def plot_training_curve(train_log: list[dict], run_id: str) -> None:
    """
    Two-panel figure:
      Left  — train_loss over epochs
      Right — BLEU-4, CIDEr, ROUGE-L over epochs
    """
    if not train_log:
        print(f"No training log for {run_id}")
        return

    epochs   = [r["epoch"]      for r in train_log]
    tr_loss  = [r["train_loss"] for r in train_log]
    bleu4    = [r["bleu_4"]     for r in train_log]
    cider    = [r["cider"]      for r in train_log]
    rouge    = [r["rouge_l"]    for r in train_log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{run_id} — training curves", fontsize=13)

    ax1.plot(epochs, tr_loss, marker="o", color="steelblue", linewidth=2)
    ax1.set_xlabel("Epoch");  ax1.set_ylabel("Train loss")
    ax1.set_title("Training loss")
    ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax1.grid(alpha=0.3)

    ax2.plot(epochs, bleu4, marker="o", label="BLEU-4",  color="#e07b39")
    ax2.plot(epochs, cider, marker="s", label="CIDEr",   color="#4caf50")
    ax2.plot(epochs, rouge, marker="^", label="ROUGE-L", color="#9c27b0")
    ax2.set_xlabel("Epoch");  ax2.set_ylabel("Score")
    ax2.set_title("Validation metrics")
    ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2.legend();  ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 8: compare_runs() — aggregated metrics table + bar chart

def compare_runs(results: list[dict]) -> pd.DataFrame:
    """
    Build a DataFrame with one row per run showing BLEU-4, CIDEr, ROUGE-L.
    Also renders a grouped bar chart.
    Sorted by CIDEr descending (best model first).
    """
    rows = []
    for r in results:
        if r is None:
            continue
        s = r["scores"]
        rows.append({
            "run_id"      : r["run_id"],
            "description" : r.get("description", ""),
            "BLEU-4"      : round(s.get("bleu_4",  -1), 4),
            "CIDEr"       : round(s.get("cider",   -1), 4),
            "ROUGE-L"     : round(s.get("rouge_l", -1), 4),
            "METEOR"      : round(s.get("meteor",  -1), 4),
        })

    if not rows:
        print("No results to compare yet.")
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values("CIDEr", ascending=False).reset_index(drop=True)

    # Bar chart
    metrics  = ["BLEU-4", "CIDEr", "ROUGE-L"]
    n_runs   = len(df)
    x        = np.arange(n_runs)
    width    = 0.22
    colors   = ["#e07b39", "#4caf50", "#9c27b0"]

    fig, ax = plt.subplots(figsize=(max(8, n_runs * 1.4), 5))
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        offsets = x + (i - 1) * width
        bars    = ax.bar(offsets, df[metric], width, label=metric, color=color, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h >= 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2, h + 0.002,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=7,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(df["run_id"].tolist(), rotation=30, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("M3 Ablation Results — sorted by CIDEr")
    ax.legend();  ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    return df[["run_id", "description", "BLEU-4", "CIDEr", "ROUGE-L", "METEOR"]]

---
## M3 Ablation Runs
Execute the cells below **in order**. Each run saves its outputs to `outputs/runs/{run_id}/`.

> **Tip:** Re-running a cell for a run that already has a cached CLIP embedding file (`outputs/cache/clip_cache_{run_id}.pt`) skips the slow embedding step — subsequent runs of the same `run_id` start immediately.

In [ ]:
#quick check

import torch
from src.decoder import PrefixProjection

# Quick gradient flow test — 5 forward+backward passes
clip_dim  = 512
gpt2_dim  = 768
num_prefix = 10

proj = PrefixProjection(clip_dim, gpt2_dim, num_prefix)
opt  = torch.optim.Adam(proj.parameters(), lr=1e-3)

for step in range(5):
    dummy_clip = torch.randn(4, clip_dim)          # batch of 4
    out = proj(dummy_clip)                          # [4, 10, 768]
    loss = out.pow(2).mean()                        # dummy loss
    loss.backward()
    # projection is nn.Sequential (MLP) — use first Linear layer for grad check
    grad_norm = proj.projection[0].weight.grad.norm().item()
    opt.step(); opt.zero_grad()
    print(f"step {step+1}: loss={loss.item():.4f} | grad_norm={grad_norm:.6f}")

print("If grad_norm > 0 on every step → gradient flow OK")

In [ ]:
# Prefix Similarity Debug Cell
# Run after at least 1 epoch to check if prefix varies per image.
# sim < 0.95 = projection is discriminating images (good)
# sim > 0.99 = projection is degenerate (all images get same caption)
import torch, torch.nn.functional as F

def check_prefix_similarity(prefix_proj, val_samples, device, n_pairs=5):
    prefix_proj.eval()
    print("=== Prefix Cosine Similarity Between Different Images ===")
    print("(< 0.95 = discriminating | > 0.99 = degenerate)")
    with torch.no_grad():
        for i in range(n_pairs):
            a = torch.tensor(val_samples[i]["clip_emb"]).unsqueeze(0).to(device)
            b = torch.tensor(val_samples[i + 50]["clip_emb"]).unsqueeze(0).to(device)
            pa = prefix_proj(a).flatten()
            pb = prefix_proj(b).flatten()
            sim = F.cosine_similarity(pa, pb, dim=0).item()
            status = "OK" if sim < 0.95 else "DEGENERATE"
            print(f"  pair {i+1}: sim={sim:.4f}  [{status}]")

# Usage: run after a training step completes
# check_prefix_similarity(prefix_proj, val_samples, device)
# Or inline after run_experiment:
# check_prefix_similarity(result_001["prefix_proj"], val_samples, device)


---
## Stage 1 — Learning Rate Sweep
Model: CLIP-L/14 + GPT-2 Medium | LoRA r=8 | Beam decoding  
Vary: `lr` ∈ {1e-4, 5e-5, 1e-5}  
**Goal:** find the best learning rate before sweeping other hyperparams.


In [ ]:
result_lr_1e4 = run_experiment(STAGE1_RUNS[0], config, device)
show_samples(result_lr_1e4, n=3)


In [ ]:
result_lr_5e5 = run_experiment(STAGE1_RUNS[1], config, device)
show_samples(result_lr_5e5, n=3)


In [ ]:
result_lr_1e5 = run_experiment(STAGE1_RUNS[2], config, device)
show_samples(result_lr_1e5, n=3)


In [ ]:
# Stage 1 comparison — pick best LR
stage1_results = [r for r in [result_lr_1e4, result_lr_5e5, result_lr_1e5] if r is not None]
compare_runs(stage1_results)
# >>> After reviewing, set BEST_LR in Cell 3 (Stage 2 block) <<<


---
## Stage 2 — LoRA Rank Sweep
Model: CLIP-L/14 + GPT-2 Medium | `BEST_LR` from Stage 1 | Beam decoding  
Vary: `lora_rank` ∈ {4, 8, 16}  
**Goal:** find the right adapter capacity for GPT-2 Medium.


In [ ]:
result_lora_r4  = run_experiment(STAGE2_RUNS[0], config, device)
show_samples(result_lora_r4, n=3)


In [ ]:
result_lora_r8  = run_experiment(STAGE2_RUNS[1], config, device)
show_samples(result_lora_r8, n=3)


In [ ]:
result_lora_r16 = run_experiment(STAGE2_RUNS[2], config, device)
show_samples(result_lora_r16, n=3)


In [ ]:
# Stage 2 comparison — pick best LoRA rank
stage2_results = [r for r in [result_lora_r4, result_lora_r8, result_lora_r16] if r is not None]
compare_runs(stage2_results)
# >>> After reviewing, set BEST_RANK in Cell 3 (Stage 3 block) <<<


---
## Stage 3 — Decoding Strategy Sweep
Model: CLIP-L/14 + GPT-2 Medium | `BEST_LR` | `BEST_RANK`  
Vary: greedy, beam k=5, nucleus (3 temp x top_p combinations)  
**Goal:** find the best decoding strategy.


In [ ]:
result_dec_greedy     = run_experiment(STAGE3_RUNS[0], config, device)
show_samples(result_dec_greedy, n=3)


In [ ]:
result_dec_beam       = run_experiment(STAGE3_RUNS[1], config, device)
show_samples(result_dec_beam, n=3)


In [ ]:
result_dec_nuc_07_09  = run_experiment(STAGE3_RUNS[2], config, device)
show_samples(result_dec_nuc_07_09, n=3)


In [ ]:
result_dec_nuc_10_09  = run_experiment(STAGE3_RUNS[3], config, device)
show_samples(result_dec_nuc_10_09, n=3)


In [ ]:
result_dec_nuc_05_095 = run_experiment(STAGE3_RUNS[4], config, device)
show_samples(result_dec_nuc_05_095, n=3)


In [ ]:
# Stage 3 comparison — pick best decoding strategy
stage3_results = [r for r in [
    result_dec_greedy, result_dec_beam,
    result_dec_nuc_07_09, result_dec_nuc_10_09, result_dec_nuc_05_095,
] if r is not None]
compare_runs(stage3_results)
# >>> If nucleus wins: set BEST_DECODER='nucleus' and run Stage 4.
# >>> If beam/greedy wins: skip Stage 4, go straight to Stage 5. <<<


---
## Stage 4 — Nucleus Fine-Tuning *(run only if nucleus won Stage 3)*
Model: CLIP-L/14 + GPT-2 Medium | `BEST_LR` | `BEST_RANK` | nucleus  
Vary: `temperature` x `top_p` grid  
**Goal:** find the best nucleus sampling parameters.


In [ ]:
result_nuc_05_08  = run_experiment(STAGE4_RUNS[0], config, device)
result_nuc_05_09  = run_experiment(STAGE4_RUNS[1], config, device)
result_nuc_07_08  = run_experiment(STAGE4_RUNS[2], config, device)
result_nuc_10_095 = run_experiment(STAGE4_RUNS[3], config, device)


In [ ]:
# Stage 4 comparison — pick best nucleus params
stage4_results = [r for r in [
    result_nuc_05_08, result_nuc_05_09, result_nuc_07_08, result_nuc_10_095,
] if r is not None]
compare_runs(stage4_results)
# >>> Set BEST_TEMP and BEST_TOP_P in Cell 3 (Stage 5 block) <<<


---
## Stage 5 — Final Best Config (Extended Training)
Uses best LR, LoRA rank, and decoding from Stages 1–4.  
Trains for 10 epochs with patience=5.  
**This is the final model checkpoint for the Streamlit demo.**


In [ ]:
result_final = run_experiment(STAGE5_RUNS[0], config, device)
show_samples(result_final, n=5)


---
## Full Sweep Summary


In [ ]:
# Collect all completed results for final cross-stage comparison
all_stage_results = []
for var_name in [
    'result_lr_1e4', 'result_lr_5e5', 'result_lr_1e5',
    'result_lora_r4', 'result_lora_r8', 'result_lora_r16',
    'result_dec_greedy', 'result_dec_beam',
    'result_dec_nuc_07_09', 'result_dec_nuc_10_09', 'result_dec_nuc_05_095',
    'result_nuc_05_08', 'result_nuc_05_09', 'result_nuc_07_08', 'result_nuc_10_095',
    'result_final',
]:
    r = locals().get(var_name) or globals().get(var_name)
    if r is not None:
        all_stage_results.append(r)

if all_stage_results:
    compare_runs(all_stage_results)
else:
    print('No results yet — run the stage cells above.')


---


In [ ]:
# Custom Experiment Template


---
